# 06 - feature engineering

takes the demand + weather parquet and builds all the features we use for modelling.

the most important ones are the demand lag features - lag_1 (1 hour ago) and lag_168 (1 week ago) end up dominating model behaviour. temporal, cyclical, rolling and spatial features round it out.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT     = Path('/Users/Xavier/Desktop/university/YEAR4/datascienceNO3')
IN_PATH  = ROOT / 'exports' / 'demand_weather_joined_2023_2024.parquet'
OUT_PATH = ROOT / 'exports' / 'demand_weather_features_2023_2024.parquet'
ADJ_PATH = ROOT / 'taxi_zones_adjacency_matrix.csv'

TARGET = 'total_trips'

WEATHER_FEATURES  = ['temperature_2m', 'precipitation', 'snowfall', 'windspeed_10m', 'weathercode']
WEATHER_LAG_STEPS = [1, 24, 168]

US_HOLIDAYS = {
    '2023-01-02','2023-01-16','2023-02-20','2023-05-29','2023-06-19',
    '2023-07-04','2023-09-04','2023-10-09','2023-11-10','2023-11-23','2023-12-25',
    '2024-01-01','2024-01-15','2024-02-19','2024-05-27','2024-06-19',
    '2024-07-04','2024-09-02','2024-10-14','2024-11-11','2024-11-28','2024-12-25',
}
HOLIDAY_DATES = pd.to_datetime(list(US_HOLIDAYS)).normalize()

print(f'loading {IN_PATH.name}...', flush=True)
df = pd.read_parquet(IN_PATH)
df['hour_slot'] = pd.to_datetime(df['hour_slot'])
df = df.sort_values(['zone_id', 'hour_slot']).reset_index(drop=True)
print(f'  {len(df):,} rows, {df["zone_id"].nunique()} zones')

## temporal features

hour, day of week, month, weekend flag, rush hour flag, public holiday flag.

In [ ]:
print('adding temporal features...', flush=True)
df['hour']        = df['hour_slot'].dt.hour.astype('int8')
df['day_of_week'] = df['hour_slot'].dt.dayofweek.astype('int8')
df['month']       = df['hour_slot'].dt.month.astype('int8')
df['is_weekend']  = df['day_of_week'].isin([5, 6]).astype('int8')
df['is_rush_hour'] = (
    df['hour'].between(6, 9) | df['hour'].between(16, 19)
).astype('int8')
df['is_public_holiday'] = (
    df['hour_slot'].dt.normalize().isin(HOLIDAY_DATES)
).astype('int8')

## cyclical encoding

hour, day of week and month are all periodic - 23:00 and 00:00 are one hour apart, not 23 apart. we encode them as sin/cos pairs so the model sees that continuity.

In [ ]:
print('adding cyclical features...', flush=True)
h_rad   = 2 * np.pi * df['hour']        / 24
dow_rad = 2 * np.pi * df['day_of_week'] / 7
mon_rad = 2 * np.pi * df['month']       / 12

df['hour_sin']  = np.sin(h_rad).astype('float32')
df['hour_cos']  = np.cos(h_rad).astype('float32')
df['dow_sin']   = np.sin(dow_rad).astype('float32')
df['dow_cos']   = np.cos(dow_rad).astype('float32')
df['month_sin'] = np.sin(mon_rad).astype('float32')
df['month_cos'] = np.cos(mon_rad).astype('float32')

## demand lag features

these are the most important features. we shift the trip count within each zone by 1, 2, 24, 168 and 336 hours. lag_1 captures short-term autocorrelation, lag_168 captures the weekly pattern.

In [ ]:
print('adding demand lag features...', flush=True)
for lag_h, col in [(1,'lag_1'),(2,'lag_2'),(24,'lag_24'),(168,'lag_168'),(336,'lag_336')]:
    df[col] = df.groupby('zone_id')[TARGET].shift(lag_h)

## weather lag features

lagged versions of each weather variable at 1, 24 and 168 hours. this lets the model test whether past weather has any additional signal beyond current weather.

In [ ]:
print('adding weather lag features...', flush=True)
for feat in WEATHER_FEATURES:
    for lag_h in WEATHER_LAG_STEPS:
        df[f'{feat}_lag_{lag_h}'] = (
            df.groupby('zone_id')[feat].shift(lag_h).astype('float32')
        )
print(f'  done ({len(WEATHER_FEATURES) * len(WEATHER_LAG_STEPS)} features)')

## rolling demand features

rolling means and std over the last 3 and 6 hours. we shift by 1 first so there's no leakage - the rolling window never includes the current hour.

In [ ]:
print('adding rolling features...', flush=True)
grp = df.groupby('zone_id')[TARGET]
df['rolling_mean_3h'] = grp.transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
).astype('float32')
df['rolling_mean_6h'] = grp.transform(
    lambda x: x.shift(1).rolling(6, min_periods=1).mean()
).astype('float32')
df['rolling_std_6h'] = grp.transform(
    lambda x: x.shift(1).rolling(6, min_periods=2).std()
).fillna(0).astype('float32')

## spatial lag features

mean demand from neighbouring zones at lag_1 and lag_24. neighbours come from a zone adjacency matrix (rook contiguity from the tlc shapefile).

In [ ]:
print('adding spatial lag features...', flush=True)
if not ADJ_PATH.exists():
    print(f'  warning: {ADJ_PATH.name} not found - skipping spatial lags')
else:
    adj_raw = pd.read_csv(ADJ_PATH, index_col=0)
    adj_raw.index   = adj_raw.index.astype(int)
    adj_raw.columns = adj_raw.columns.astype(int)
    neighbours = {z: list(adj_raw.columns[adj_raw.loc[z] > 0]) for z in adj_raw.index}

    demand_pivot = df.pivot(index='hour_slot', columns='zone_id', values=TARGET)
    lag1_piv     = demand_pivot.shift(1)
    lag24_piv    = demand_pivot.shift(24)

    sl1, sl24 = {}, {}
    for z in demand_pivot.columns:
        if z in neighbours:
            nbrs   = [n for n in neighbours[z] if n in demand_pivot.columns]
            sl1[z]  = lag1_piv[nbrs].mean(axis=1)  if nbrs else pd.Series(np.nan, index=demand_pivot.index)
            sl24[z] = lag24_piv[nbrs].mean(axis=1) if nbrs else pd.Series(np.nan, index=demand_pivot.index)

    def pivot_to_df(d, col):
        return (pd.DataFrame(d).stack().reset_index()
                  .rename(columns={'level_0': 'hour_slot', 'level_1': 'zone_id', 0: col}))

    df = df.merge(pivot_to_df(sl1,  'spatial_lag_1'),  on=['hour_slot', 'zone_id'], how='left')
    df = df.merge(pivot_to_df(sl24, 'spatial_lag_24'), on=['hour_slot', 'zone_id'], how='left')

drop the first couple of weeks where lags are nan (the model can't learn from those rows anyway), then save.

In [ ]:
lag_cols = ['lag_1','lag_2','lag_24','lag_168','lag_336',
            'spatial_lag_1','spatial_lag_24',
            'rolling_mean_3h','rolling_mean_6h']
lag_cols = [c for c in lag_cols if c in df.columns]

before = len(df)
df = df.dropna(subset=lag_cols).reset_index(drop=True)
print(f'dropped {before - len(df):,} nan rows from lag warm-up -> {len(df):,} rows remain')

OUT_PATH.parent.mkdir(exist_ok=True)
df.to_parquet(OUT_PATH, index=False)

print(f'saved -> {OUT_PATH}')
print(f'  shape : {df.shape}')
print(f'  zones : {df["zone_id"].nunique()}')
print(f'  cols  : {df.columns.tolist()}')